# Step 8 · Score applicable pathogen features

Score every question in the saved host/pathogen trait profile against every accepted pathogen–host abstract from the run. Pathogen, host, and pair scope are reflected in Jev's question state and decision cache. The step resumes from its feature decision ledger and fails visibly when scoring is incomplete.

In [1]:
import sys, json, yaml
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id
from pathogen_atlas_v4.pubmed_client import literature_cache_key
from pathogen_atlas_v4.feature_screening import build_feature_screen_tasks
from pathogen_atlas_v4.decision_runner import run_decisions
from pathogen_atlas_v4.jev_pathogen_screen import make_jev_feature_screen_handler, openrouter_key_available
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('08_score_feature_profiles', RUN_ID)
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
RUN_CONFIG = json.loads((RUN_DIR / 'resolved_config.json').read_text(encoding='utf-8'))
CONFIG = RUN_CONFIG['source_config']
STAGING_TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
STAGING_ROOT = STAGING_TAXONOMY_ROOT.parent
PATHOGENS = pd.read_parquet(RUN_DIR / 'taxonomy/pathogens.parquet')
HOSTS = pd.read_parquet(STAGING_TAXONOMY_ROOT / 'hosts.parquet')
PAIR_SELECTION = pd.read_parquet(RUN_DIR / 'pathogen_host_selection.parquet')
PROFILE_DIR = RUN_DIR / 'feature_profiles'
PROFILES = [{'resolved_profile': json.loads(path.read_text(encoding='utf-8'))} for path in sorted(PROFILE_DIR.glob('*.json'))]
PROMPTS = yaml.safe_load((ROOT / CONFIG['decisions']['prompt_catalogue']).read_text(encoding='utf-8'))
CACHE_KEY = literature_cache_key(query_version=CONFIG['acquisition']['query_strategy_version'],
    pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
    taxonomy_snapshot=STAGING_ROOT.name)
CACHE_DIR = ROOT / 'data/v4/cache/literature' / CACHE_KEY
SAMPLE_MATCHES_PATH = CACHE_DIR / 'tables/pathogen_screen_matches.parquet'
ARTICLES_PATH = CACHE_DIR / 'tables/articles.parquet'
if not SAMPLE_MATCHES_PATH.exists() or not ARTICLES_PATH.exists():
    raise FileNotFoundError(f'No completed pair sample cache for this run: {CACHE_DIR}. Run notebook 03, then notebook 04.')
SAMPLE_MATCHES = pd.read_parquet(SAMPLE_MATCHES_PATH)
ARTICLES = pd.read_parquet(ARTICLES_PATH)
SELECTED_PAIRS = set(zip(PAIR_SELECTION.pathogen_id.astype(str), PAIR_SELECTION.host_id.astype(str)))
SAMPLE_MATCHES = SAMPLE_MATCHES.loc[[key in SELECTED_PAIRS for key in zip(SAMPLE_MATCHES.pathogen_id.astype(str), SAMPLE_MATCHES.host_id.astype(str))]].copy()
print('Run:', RUN_ID, '| cache:', CACHE_KEY, '| profiles:', len(PROFILES), '| selected sample rows:', len(SAMPLE_MATCHES))
LOG.info('Feature score inputs: sample_rows=%d profiles=%d cache=%s', len(SAMPLE_MATCHES), len(PROFILES), CACHE_KEY)

2026-11-03 00:33 INFO Notebook session started at 2026-11-03T00:33:19.352214+00:00


2026-11-03 00:33 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-03 00:33 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/08_score_feature_profiles.log


2026-11-03 00:33 INFO Feature score inputs: sample_rows=14 profiles=4 cache=965b253d5343fc060069ccda62a1cb196b01997fa67e29a22e5a6db9471caf9c


Run: pilot-v1 | cache: 965b253d5343fc060069ccda62a1cb196b01997fa67e29a22e5a6db9471caf9c | profiles: 4 | selected sample rows: 14


## Build the complete applicable task set

Only accepted pathogen–host evidence abstracts enter feature scoring. Each resolved question produces a task for every accepted abstract. Missing abstracts are not scored and are excluded from the denominator.

In [2]:
FEATURE_PROFILE_VERSION = str(CONFIG.get('features', {}).get('profile_version', ''))
MODEL_VERSION = str(CONFIG.get('decisions', {}).get('model_version', 'typesafe/jev-1.14'))
FEATURE_TASKS = build_feature_screen_tasks(SAMPLE_MATCHES, ARTICLES, PAIR_SELECTION, PATHOGENS, HOSTS,
    PROFILES, PROMPTS, run_id=RUN_ID,
    run_config_version=str(RUN_CONFIG.get('config_version', CONFIG.get('run', {}).get('config_version', ''))),
    model_version=MODEL_VERSION, profile_version=FEATURE_PROFILE_VERSION)
if not FEATURE_TASKS:
    if PAIR_SELECTION.selection_status.astype(str).eq('accepted').any():
        raise RuntimeError('Accepted pathogen–host pairs exist, but no applicable feature tasks were produced. Check profiles and accepted evidence PMID samples.')
    print('No accepted pathogen–host evidence is available to score.')
else:
    coverage = pd.DataFrame(FEATURE_TASKS).groupby(['pathogen_id','host_id','feature_id'], dropna=False).pmid.nunique().rename('abstracts').reset_index()
    print('Feature tasks:', f'{len(FEATURE_TASKS):,}', '| accepted pairs:', PAIR_SELECTION.selection_status.astype(str).eq('accepted').sum(), '| features:', coverage.feature_id.nunique())
    display(coverage.groupby('host_id').agg(features=('feature_id','nunique'), pair_features=('pathogen_id','nunique'), tasks=('abstracts','sum')))
    LOG.info('Built complete applicable feature tasks=%d', len(FEATURE_TASKS))

Feature tasks: 200 | accepted pairs: 3 | features: 40


,features,pair_features,tasks
host_id,,,
ncbi:4530,37,3,185


2026-11-03 00:33 INFO Built complete applicable feature tasks=200


## Score with Jev and verify completion

The checkpoint ledger makes this resumable. Keep the worker count moderate for the provider; completed decisions are reused on later runs with the same scoped question and abstract.

In [3]:
FEATURE_LEDGER_PATH = RUN_DIR / 'feature_screen_ledger.parquet'
FEATURE_RAW_PATH = RUN_DIR / 'feature_screen_raw_responses.jsonl'
if FEATURE_TASKS:
    if not openrouter_key_available():
        raise RuntimeError('Feature scoring requires OPENROUTER_API_KEY or an OpenRouter Keychain entry.')
    FEATURE_LEDGER = run_decisions(FEATURE_TASKS, make_jev_feature_screen_handler(model=MODEL_VERSION),
        ledger_path=FEATURE_LEDGER_PATH, raw_response_path=FEATURE_RAW_PATH,
        max_attempts=int(CONFIG.get('logging', {}).get('max_attempts', 2)),
        max_workers=min(6, int(CONFIG.get('logging', {}).get('decision_workers', 6))), checkpoint_every=20)
    task_keys = {task['cache_key'] for task in FEATURE_TASKS}
    current = FEATURE_LEDGER.loc[FEATURE_LEDGER.cache_key.astype(str).isin(task_keys)]
    completed = current.loc[current.status.astype(str).eq('completed')]
    missing = task_keys - set(completed.cache_key.astype(str))
    if missing:
        raise RuntimeError(f'Feature scoring incomplete: {len(missing)} of {len(task_keys)} applicable questions remain unscored.')
    print('All applicable feature questions scored:', f'{len(completed):,}', '| ledger:', FEATURE_LEDGER_PATH)
    LOG.info('Feature scoring complete tasks=%d unique_completed=%d', len(FEATURE_TASKS), len(completed))
else:
    FEATURE_LEDGER = pd.read_parquet(FEATURE_LEDGER_PATH) if FEATURE_LEDGER_PATH.exists() else pd.DataFrame()

2026-11-03 00:33 INFO Feature scoring complete tasks=200 unique_completed=200


All applicable feature questions scored: 200 | ledger: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/feature_screen_ledger.parquet
